In [1]:
import sympleq

# SympleQ demo: Clifford simulation on thousands of qubits

In this notebook we show a different kind of classical simulation.

Instead of storing the quantum statevector, we track how Pauli operators transform under a Clifford circuit.

For a Clifford unitary \(U\), a Pauli operator \(P\) transforms as

$$
P \mapsto U^\dagger P U .
$$

The result is still a Pauli operator.

This is the algebraic reason Clifford circuits can be simulated efficiently.

In [19]:
import time
import numpy as np
import matplotlib.pyplot as plt

from sympleq.core.circuits import Circuit, GATES
from sympleq.core.paulis import PauliSum, PauliString

In [20]:
H = GATES.H
S = GATES.S
CX = GATES.CX
CZ = GATES.CZ

## 1. A small PauliSum example

A PauliString is one product of Pauli operators.

A PauliSum is a weighted sum of PauliStrings.

For example,

$$
A = 1.0 Z_0 + 0.5 X_1 - 1.2 Z_0 Z_1 .
$$

We will construct this directly using `PauliString.from_exponents(...)` and `PauliSum.from_pauli_strings(...)`.

In [21]:
dimensions = [2, 2, 2]

z0 = PauliString.from_exponents(
    x_exp=[0, 0, 0],
    z_exp=[1, 0, 0],
    dimensions=dimensions,
)

x1 = PauliString.from_exponents(
    x_exp=[0, 1, 0],
    z_exp=[0, 0, 0],
    dimensions=dimensions,
)

z0z1 = PauliString.from_exponents(
    x_exp=[0, 0, 0],
    z_exp=[1, 1, 0],
    dimensions=dimensions,
)

A = PauliSum.from_pauli_strings(
    [z0, x1, z0z1],
    weights=[1.0, 0.5, -1.2],
)

print(A)

(1+0j)   |x0z1 x0z0 x0z0 | 0 
(0.5+0j) |x0z0 x1z0 x0z0 | 0 
(-1.2+0j)|x0z1 x0z1 x0z0 | 0 



## 2. Inspect the PauliSum data

For \(k\) Pauli terms on \(n\) qubits, the tableau has shape

$$
(k, 2n).
$$

The first \(n\) columns store the \(X\)-exponents.

The last \(n\) columns store the \(Z\)-exponents.

So each row looks like

$$
[x_0, x_1, \dots, x_{n-1} \mid z_0, z_1, \dots, z_{n-1}].
$$

The external row on the left stores the co-efficients, and the right stores the phase values

In [23]:
print("Tableau:")
print(A.tableau)

print("\nWeights:")
print(A.weights)

print("\nPhases:")
print(A.phases)

print("\nDimensions:")
print(A.dimensions)

print("\nTableau shape:")
print(A.tableau.shape)

Tableau:
[[0 0 0 1 0 0]
 [0 1 0 0 0 0]
 [0 0 0 1 1 0]]

Weights:
[ 1. +0.j  0.5+0.j -1.2+0.j]

Phases:
[0 0 0]

Dimensions:
[2 2 2]

Tableau shape:
(3, 6)


## 2. Act on a PauliSum with a Clifford circuit

Now we build a small Clifford circuit and act on the PauliSum.

Conceptually, we are transforming the operator under the Clifford circuit.

For a Clifford unitary \(U\), a Pauli operator transforms to another Pauli operator:

$$
P \mapsto U^\dagger P U .
$$

So a PauliSum transforms term by term into another PauliSum.

In [30]:
qc = Circuit.from_tuples(
    dimensions=[2, 2, 2],
    data=[
        (H, 0),
        (CX, 0, 1),
        (S, 2),
    ],
)

print(qc)

Circuit on 3 qudits (dims=[np.int64(2), np.int64(2), np.int64(2)]):
  H (0,) 
  CX (0, 1) 
  S (2,) 


In [31]:
A_transformed = qc.act(A)

print(A_transformed)

(1+0j)   |x1z0 x1z0 x0z0 | 0 
(0.5+0j) |x0z0 x1z0 x0z0 | 0 
(-1.2+0j)|x1z1 x1z1 x0z0 | 0 



In [32]:
print("Original tableau:")
print(A.tableau)

print("\nTransformed tableau:")
print(A_transformed.tableau)

Original tableau:
[[0 0 0 1 0 0]
 [0 1 0 0 0 0]
 [0 0 0 1 1 0]]

Transformed tableau:
[[1 1 0 0 0 0]
 [0 1 0 0 0 0]
 [1 1 0 1 1 0]]


## 4. Scaling Up with 1000 of qubits

The previous example used only 3 qubits.

Now we use the same idea for many qubits.

We will build a Clifford circuit on thousands of qubits and propagate a small PauliSum through it.

The important point is that we are not storing a statevector.

For one PauliString on \(n\) qubits, the tableau has only \(2n\) entries.

In [ ]:
def make_large_pauli_sum(n_qubits):
    """
    Create a simple PauliSum on n_qubits:

        A = Z_0 + 0.5 X_1 - 1.2 Z_0 Z_1 + x_0 x_1

    The operator acts nontrivially only on the first two qubits,
    but it is represented as an n-qubit PauliSum.
    """
    dimensions = np.full(n_qubits, 2, dtype=int)

    z0 = PauliString.from_exponents(
        x_exp=np.zeros(n_qubits, dtype=int),
        z_exp=np.eye(1, n_qubits, 0, dtype=int).ravel(),
        dimensions=dimensions,
    )

    x1_x = np.zeros(n_qubits, dtype=int)
    x1_x[1] = 1

    x1 = PauliString.from_exponents(
        x_exp=x1_x,
        z_exp=np.zeros(n_qubits, dtype=int),
        dimensions=dimensions,
    )

    z0z1_z = np.zeros(n_qubits, dtype=int)
    z0z1_z[0] = 1
    z0z1_z[1] = 1

    z0z1 = PauliString.from_exponents(
        x_exp=np.zeros(n_qubits, dtype=int),
        z_exp=z0z1_z,
        dimensions=dimensions,
    )

    x0x1_x = np.zeros(n_qubits, dtype=int)
    x0x1_x[0] = 1
    x0x1_x[1] = 1

    x0x1 = PauliString.from_exponents(
        x_exp=x0x1_x,
        z_exp=np.zeros(n_qubits, dtype=int),
        dimensions=dimensions,
    )

    return PauliSum.from_pauli_strings(
        [z0, x1, z0z1],
        weights=[1.0, 0.5, -1.2],
    )

In [35]:
def clifford_chain_circuit(n_qubits, depth=3):
    """
    Build a simple 1D Clifford circuit on n_qubits.

    The circuit contains local H/S gates and nearest-neighbour CX gates.
    """
    dimensions = np.full(n_qubits, 2, dtype=int)
    gates = []
    qudits = []

    for layer in range(depth):
        # Local Clifford layer
        for q in range(n_qubits):
            if (q + layer) % 2 == 0:
                gates.append(H)
                qudits.append((q,))
            else:
                gates.append(S)
                qudits.append((q,))

        # Nearest-neighbour CX brickwork
        offset = layer % 2
        for q in range(offset, n_qubits - 1, 2):
            gates.append(CX)
            qudits.append((q, q + 1))

    return Circuit.from_gates_and_qudits(
        dimensions=dimensions,
        gates=gates,
        qudit_indices=qudits,
    )

In [39]:
N = 1000
DEPTH = 100

A_large = make_large_pauli_sum(N)
qc_large = clifford_chain_circuit(N, depth=DEPTH)

print("Number of qubits:", N)
print("Number of Pauli terms:", A_large.tableau.shape[0])
print("PauliSum tableau shape:", A_large.tableau.shape)
print("Number of Clifford gates:", qc_large.n_gates())

Number of qubits: 1000
Number of Pauli terms: 3
PauliSum tableau shape: (3, 2000)
Number of Clifford gates: 149950


## 3. Propagate one observable through a 10,000-qubit circuit

Now we track

$$
Z_0 \mapsto U^\dagger Z_0 U.
$$

This is not a statevector simulation.

We are not storing \(2^{10000}\) amplitudes.

We are storing one Pauli row of length \(2n\).

In [14]:
N = 1000
DEPTH = 3

observable = single_pauli(N, kind="Z", qubit=0)
circuit = clifford_chain_circuit(N, depth=DEPTH)

print("Number of qubits:", N)
print("Number of gates:", circuit.n_gates())
print("Observable tableau shape:", observable.tableau.shape)

Number of qubits: 1000
Number of gates: 4499
Observable tableau shape: (1, 2000)


In [15]:
start = time.perf_counter()
out_observable = circuit.act(observable)
elapsed = time.perf_counter() - start

print(f"Propagation time: {elapsed:.3f} seconds")
print("Output tableau shape:", out_observable.tableau.shape)
print("Number of nonzero entries in output Pauli:")
print(np.count_nonzero(out_observable.tableau))

Propagation time: 0.971 seconds
Output tableau shape: (1, 2000)
Number of nonzero entries in output Pauli:
3


In [16]:
out_observable.tableau

array([[1, 1, 0, ..., 0, 0, 0]], shape=(1, 2000))

## What did we just simulate?

We simulated the Clifford evolution of an observable through a 1000-qubit circuit.

A statevector simulator would need \(2^{1000}\) amplitudes, which is impossible.

But SympleQ only tracks the Pauli tableau.

For one Pauli observable, that is a \(1 \times 2n\) binary/integer object.